In [1]:
from pathlib import Path
import sys
import copy
import random

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Work\Quantum-Adversarial-Robustness


In [2]:
SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

print("Seed:", SEED)

Seed: 42


In [3]:
NUM_QUBITS = 4

BATCH_SIZE = 32
EPOCHS = 30

LEARNING_RATE = 0.01

DEVICE = torch.device("cpu")

print("=" * 50)
print("NOTEBOOK 04A — CLEAN VQC BASELINE")
print("=" * 50)

print("Seed       :", SEED)
print("Qubits     :", NUM_QUBITS)
print("Batch size :", BATCH_SIZE)
print("Epochs     :", EPOCHS)
print("LR         :", LEARNING_RATE)
print("Device     :", DEVICE)

NOTEBOOK 04A — CLEAN VQC BASELINE
Seed       : 42
Qubits     : 4
Batch size : 32
Epochs     : 30
LR         : 0.01
Device     : cpu


In [4]:
DATA_DIR = PROJECT_ROOT / "data" / "binary"

X_train = np.load(DATA_DIR / "X_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")

y_train = np.load(DATA_DIR / "y_train.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (11824, 4)
y_train: (11824,)
X_test : (2956, 4)
y_test : (2956,)


In [5]:
assert X_train.ndim == 2
assert X_test.ndim == 2

assert X_train.shape[1] == 4
assert X_test.shape[1] == 4

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("Dataset validation passed.")

Dataset validation passed.


In [6]:
assert X_train.ndim == 2
assert X_test.ndim == 2

assert X_train.shape[1] == 4
assert X_test.shape[1] == 4

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("Dataset validation passed.")

Dataset validation passed.


In [7]:
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train,
)

print("Training samples  :", len(X_train_split))
print("Validation samples:", len(X_val))
print("Test samples      :", len(X_test))

Training samples  : 9459
Validation samples: 2365
Test samples      : 2956


In [8]:
X_train_tensor = torch.tensor(
    X_train_split,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_split,
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

print(X_train_tensor.shape)
print(y_train_tensor.shape)
print(X_val_tensor.shape)
print(y_val_tensor.shape)
print(X_test_tensor.shape)
print(y_test_tensor.shape)

torch.Size([9459, 4])
torch.Size([9459, 1])
torch.Size([2365, 4])
torch.Size([2365, 1])
torch.Size([2956, 4])
torch.Size([2956, 1])


In [9]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("DataLoaders created.")

DataLoaders created.


In [10]:
from src.models.quantum_model import create_model
from src.models.hybrid_classifier import HybridClassifier
from src.training.trainer import Trainer

In [11]:
quantum_model = create_model(
    num_qubits=NUM_QUBITS
)

model = HybridClassifier(
    quantum_model=quantum_model
)

model = model.to(DEVICE)

print(model)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [12]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Criterion:", criterion)
print("Optimizer:", optimizer)

Criterion: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)


In [13]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
)

In [14]:
import time

start_time = time.perf_counter()

history_04A = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
)

elapsed = time.perf_counter() - start_time

print()
print("=" * 50)
print("04A TRAINING COMPLETE")
print("=" * 50)
print(f"Training time: {elapsed / 60:.2f} minutes")

Epoch 001/030 | Train Loss: 0.6904 | Val Loss: 0.6857 | Train Acc: 0.5588 | Val Acc: 0.5378
Epoch 002/030 | Train Loss: 0.6799 | Val Loss: 0.6797 | Train Acc: 0.5692 | Val Acc: 0.5522
Epoch 003/030 | Train Loss: 0.6775 | Val Loss: 0.6800 | Train Acc: 0.5680 | Val Acc: 0.5594
Epoch 004/030 | Train Loss: 0.6753 | Val Loss: 0.6818 | Train Acc: 0.5685 | Val Acc: 0.5573
Epoch 005/030 | Train Loss: 0.6733 | Val Loss: 0.6769 | Train Acc: 0.5891 | Val Acc: 0.5691
Epoch 006/030 | Train Loss: 0.6709 | Val Loss: 0.6724 | Train Acc: 0.5840 | Val Acc: 0.5856
Epoch 007/030 | Train Loss: 0.6708 | Val Loss: 0.6725 | Train Acc: 0.5807 | Val Acc: 0.5911
Epoch 008/030 | Train Loss: 0.6704 | Val Loss: 0.6730 | Train Acc: 0.5825 | Val Acc: 0.5797
Epoch 009/030 | Train Loss: 0.6709 | Val Loss: 0.6718 | Train Acc: 0.5816 | Val Acc: 0.5839
Epoch 010/030 | Train Loss: 0.6705 | Val Loss: 0.6716 | Train Acc: 0.5827 | Val Acc: 0.5920
Epoch 011/030 | Train Loss: 0.6704 | Val Loss: 0.6721 | Train Acc: 0.5824 | Val 

In [15]:
test_loss_04A, test_metrics_04A = trainer.evaluate(
    test_loader
)

print("=" * 50)
print("04A — CLEAN VQC TEST RESULTS")
print("=" * 50)

print(f"Test loss      : {test_loss_04A:.4f}")
print(f"Test accuracy  : {test_metrics_04A['accuracy']:.4f}")
print(f"Test precision : {test_metrics_04A['precision']:.4f}")
print(f"Test recall    : {test_metrics_04A['recall']:.4f}")
print(f"Test F1        : {test_metrics_04A['f1']:.4f}")

04A — CLEAN VQC TEST RESULTS
Test loss      : 0.6707
Test accuracy  : 0.5795
Test precision : 0.5868
Test recall    : 0.7124
Test F1        : 0.6435


In [16]:
MODEL_DIR = PROJECT_ROOT / "results" / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_04A_PATH = MODEL_DIR / "vqc_04A_raw_seed42.pt"

torch.save(
    model.state_dict(),
    MODEL_04A_PATH
)

print("Saved:")
print(MODEL_04A_PATH)

Saved:
c:\Work\Quantum-Adversarial-Robustness\results\models\vqc_04A_raw_seed42.pt
